In [1]:
%pip install langchain-ollama


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install pandas langchain faiss-cpu sentence-transformers ollama streamlit langchain_huggingface langchain-ollama

   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 9.1/9.1 MB 43.6 MB/s  0:00:00
   ---------------------------------------- 0.0/795.4 kB ? eta -:--:--
   ---------------------------------------- 795.4/795.4 kB 35.6 MB/s  0:00:00
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   ---------------------------------------- 6.9/6.9 MB 47.4 MB/s  0:00:00
   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   -------------- ------------------------- 10.0/27.5 MB 47.8 MB/s eta 0:00:01
   --------------------------- ------------ 18.9/27.5 MB 45.8 MB/s eta 0:00:01
   ---------------------------------------  27.5/27.5 MB 44.8 MB/s eta 0:00:01
   ---------------------------------------- 27.5/27.5 MB 41.6 MB/s  0:00:00

   ----------------------------------------  0/11 [watchdog]
   ----------------------------------------  0/11 [watchdog]
   ----------------------------------------  0/11 [watchdog


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import ast
import langchain

In [4]:
from langchain_community.llms import Ollama
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
# from langchain.chains import RetrievalQA

from langchain_core.documents import Document

In [5]:
df = pd.read_csv("LLM Preprocessed Data.csv")

In [6]:

documents = []

for _, row in df.iterrows():
    content = f"""
Disease: {row['Disease']}
Description: {row['Description']}
Diet: {row['Diet']}
Medication: {row['Medication']}
Workout: {row['workout']}
Precautions: {row['Precautions']}
"""
    documents.append(Document(page_content=content))

In [7]:

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(documents, embeddings)

KeyboardInterrupt: 

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

In [ ]:

llm = Ollama(model="gemma3:1b")


In [ ]:
def get_recommendation(disease):
    docs = retriever.get_relevant_documents(disease)

    if not docs:
        return "No data found for this disease."

    context = docs[0].page_content

    prompt = f"""
You are a medical assistant chatbot.
Use ONLY the information in the context.
Do NOT add new medications.
Do NOT give dosage.
Add a medical disclaimer.

Context:
{context}

Provide response in this format:
- Description
- Diet
- Medication
- Workout / Lifestyle
- Precautions
- Disclaimer
"""

    return llm(prompt)


In [ ]:
docs = retriever.invoke('Precautions of Fungal Infection')


In [ ]:
docs

[Document(id='6ffb9021-91c4-43d8-944a-d806e285503e', metadata={}, page_content="\nDisease: Fungal infection\nDescription: Fungal infection is a common skin condition caused by fungi.\nDiet: Antifungal Diet', 'Probiotics', 'Garlic', 'Coconut oil', 'Turmeric'\nMedication: Antifungal Cream', 'Fluconazole', 'Terbinafine', 'Clotrimazole', 'Ketoconazole'\nWorkout: Eat foods rich in zinc', 'Include turmeric in diet', 'Consume green tea', 'Consume probiotics', 'Stay hydrated', 'Increase intake of garlic', 'Limit processed foods', 'Avoid sugary foods', 'Eat fruits and vegetables', 'Include yogurt in diet'\nPrecautions: bath twice', 'use clean cloths', 'keep infected area dry', 'use detol or neem in bathing water'\n")]

In [ ]:
chat_history = []

def chatbot(user_input):
    docs = retriever.invoke(user_input)

    if not docs:
        return "Sorry, I don't have information on that disease."

    context = docs[0].page_content

    history_text = ""
    for msg in chat_history:
        history_text += f"{msg['role']}: {msg['content']}\n"

    prompt = f"""
You are a medical assistant chatbot.
Use ONLY the provided context.
Do NOT invent medications.
Do NOT give dosage.
Always add a medical disclaimer.

Conversation History:
{history_text}

Context:
{context}

User Question:
{user_input}


"""

    response = llm.invoke(prompt)

    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": response})

    return response

In [ ]:
print("Medical Chatbot (type 'exit' to quit)\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    reply = chatbot(user_input)
    print("\nBot:", reply)

🩺 Medical Chatbot (type 'exit' to quit)



You:  list the precautions i can take for fungal infection



Bot: Okay, here are the precautions you can take for a fungal infection, according to the provided context:

*   Bath twice
*   Use clean cloths
*   Keep infected area dry
*   Use detol or neem in bathing water

**Disclaimer:** *I am providing this information based solely on the provided context and do not constitute medical advice. It’s important to consult with a healthcare professional for diagnosis and treatment of any medical condition. Self-treating can be risky and may not be appropriate. This information is for general knowledge and informational purposes only, and does not represent a definitive medical opinion.*


You:  list the workout i can do for fungal infection



Bot: Okay, here’s some workout information based on the context provided:

*   Eat foods rich in zinc
*   Include turmeric in diet
*   Consume green tea
*   Consume probiotics
*   Stay hydrated
*   Increase intake of garlic
*   Limit processed foods
*   Avoid sugary foods
*   Eat fruits and vegetables
*   Include yogurt in diet

**Disclaimer:** *I am providing this information based solely on the provided context and do not constitute medical advice. It’s important to consult with a healthcare professional for diagnosis and treatment of any medical condition. Self-treating can be risky and may not be appropriate. This information is for general knowledge and informational purposes only, and does not represent a definitive medical opinion.*
